# RAG Pipeline
## Data Ingestion to VectorDB

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

c:\Users\PI20587195\OneDrive - Wipro\Documents\banking-application\rag-chatbot\chatbot_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from langchain_core.documents import Document
import os

def process_all_texts(text_directory):
    """Process all .txt files in a directory"""
    all_documents = []
    text_dir = Path(text_directory)
    text_files = list(text_dir.glob("**/*.txt"))
    print(f"Found {len(text_files)} text files to process")
    for text_file in text_files:
        print(f"\nProcessing: {text_file.name}")
        try:
            with open(text_file, 'r', encoding='utf-8') as f:
                content = f.read()
            doc = Document(
                page_content=content,
                metadata={
                    'source_file': text_file.name,
                    'file_type': 'txt'
                }
            )
            all_documents.append(doc)
            print(f"  ✓ Loaded text file")
        except Exception as e:
            print(f"  ✗ Error: {e}")
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all text files in the data directory
all_text_documents = process_all_texts("../data")

Found 8 text files to process

Processing: KB_ACCOUNT_CREATION_REAL.txt
  ✓ Loaded text file

Processing: KB_ADMIN_GUIDE_REAL.txt
  ✓ Loaded text file

Processing: KB_FAQ_REAL.txt
  ✓ Loaded text file

Processing: KB_FEATURE_OVERVIEW_REAL.txt
  ✓ Loaded text file

Processing: KB_GETTING_STARTED_REAL.txt
  ✓ Loaded text file

Processing: KB_TROUBLESHOOTING_REAL.txt
  ✓ Loaded text file

Processing: USER_GUIDE.txt
  ✓ Loaded text file

Processing: VALIDATION_GUIDE.txt
  ✓ Loaded text file

Total documents loaded: 8


In [3]:
all_text_documents

[Document(metadata={'source_file': 'KB_ACCOUNT_CREATION_REAL.txt', 'file_type': 'txt'}, page_content='# InterBankHub - Account Creation by Country\n\nComplete step-by-step guide for creating bank accounts in India, USA, and UK.\n\n---\n\n## 🇮🇳 India - Account Creation\n\n### Account Types Available:\n- **Savings Account** - For personal saving\n- **Current Account** - For business transactions\n- **Salary Account** - Special account for salaried employees\n- **Fixed Deposit** - For fixed returns on savings\n\n### Form Fields by Section:\n\n**Step 1: Select Branch**\n- Choose your bank branch from the dropdown\n\n**Step 2: Personal Details**\n- Full Name (as per ID)\n- Date of Birth\n- Gender (Male/Female/Other)\n- Mobile Number\n- Email Address\n- Address (complete residential address)\n- Aadhaar Number (12-digit ID)\n- PAN Number (10-digit tax ID)\n\n**Step 3: Educational Details**\n- Education Level (High School, Diploma, Graduate, Post Graduate, Professional)\n- Institution Name\n- 

In [14]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=600,chunk_overlap=50):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [15]:
# Use a smaller chunk size to avoid exceeding embedding model context length
chunks=split_documents(all_text_documents, chunk_size=600, chunk_overlap=50)
chunks

Split 8 documents into 196 chunks

Example chunk:
Content: # InterBankHub - Account Creation by Country

Complete step-by-step guide for creating bank accounts in India, USA, and UK.

---

## 🇮🇳 India - Account Creation

### Account Types Available:
- **Savin...
Metadata: {'source_file': 'KB_ACCOUNT_CREATION_REAL.txt', 'file_type': 'txt'}


[Document(metadata={'source_file': 'KB_ACCOUNT_CREATION_REAL.txt', 'file_type': 'txt'}, page_content='# InterBankHub - Account Creation by Country\n\nComplete step-by-step guide for creating bank accounts in India, USA, and UK.\n\n---\n\n## 🇮🇳 India - Account Creation\n\n### Account Types Available:\n- **Savings Account** - For personal saving\n- **Current Account** - For business transactions\n- **Salary Account** - Special account for salaried employees\n- **Fixed Deposit** - For fixed returns on savings\n\n### Form Fields by Section:\n\n**Step 1: Select Branch**\n- Choose your bank branch from the dropdown'),
 Document(metadata={'source_file': 'KB_ACCOUNT_CREATION_REAL.txt', 'file_type': 'txt'}, page_content='**Step 2: Personal Details**\n- Full Name (as per ID)\n- Date of Birth\n- Gender (Male/Female/Other)\n- Mobile Number\n- Email Address\n- Address (complete residential address)\n- Aadhaar Number (12-digit ID)\n- PAN Number (10-digit tax ID)\n\n**Step 3: Educational Details**\n-

### Embedding and VectoStoreDB

In [16]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [20]:
import requests
import numpy as np

class EmbeddingManager:
    """Handles document embedding generation using Ollama API"""

    def __init__(
        self,
        ollama_url: str = "http://localhost:11434/api/embeddings",
        model: str = "all-minilm",
        max_input_chars: int = 1200,
        min_input_chars: int = 120
    ):
        """
        Initialize the embedding manager for Ollama

        Args:
            ollama_url: URL of the Ollama embeddings API endpoint
            model: Name of the embedding model to use
            max_input_chars: Target max chars per embedding request
            min_input_chars: Smallest split size before hard truncation
        """
        self.ollama_url = ollama_url
        self.model = model
        self.max_input_chars = max_input_chars
        self.min_input_chars = min_input_chars

    def _split_for_embedding(self, text: str, max_chars: int | None = None) -> list[str]:
        normalized_text = (text or "").strip()
        if not normalized_text:
            return [" "]

        chunk_limit = max_chars or self.max_input_chars
        if len(normalized_text) <= chunk_limit:
            return [normalized_text]

        parts = []
        start = 0
        overlap = min(100, max(0, chunk_limit // 10))
        step = max(1, chunk_limit - overlap)

        while start < len(normalized_text):
            end = min(start + chunk_limit, len(normalized_text))
            parts.append(normalized_text[start:end])
            if end >= len(normalized_text):
                break
            start += step

        return parts

    def _is_context_length_error(self, response: requests.Response) -> bool:
        if response.status_code != 500:
            return False
        text = (response.text or "").lower()
        return "context length" in text or "input length exceeds" in text

    def _request_embedding(self, text: str) -> list[float]:
        payload = {
            "model": self.model,
            "prompt": text
        }
        response = requests.post(self.ollama_url, json=payload, timeout=60)
        try:
            response.raise_for_status()
        except requests.HTTPError:
            if self._is_context_length_error(response) and len(text) > self.min_input_chars:
                tighter_limit = max(self.min_input_chars, len(text) // 2)
                sub_parts = self._split_for_embedding(text, max_chars=tighter_limit)
                sub_embeddings = [self._request_embedding(part) for part in sub_parts]
                if len(sub_embeddings) == 1:
                    return sub_embeddings[0]
                return np.mean(np.array(sub_embeddings), axis=0).tolist()

            print(f"HTTPError: {response.status_code} for {self.ollama_url}")
            print(f"Response content: {response.text}")
            raise

        data = response.json()
        if "embedding" not in data:
            raise ValueError("No embedding returned from Ollama")
        return data["embedding"]

    def generate_embeddings(self, texts: list) -> np.ndarray:
        """
        Generate embeddings for a list of texts using Ollama API

        Args:
            texts: List of text strings to embed
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        embeddings = []
        for idx, text in enumerate(texts, start=1):
            text_parts = self._split_for_embedding(text)
            part_embeddings = [self._request_embedding(part) for part in text_parts]

            if len(part_embeddings) == 1:
                embeddings.append(part_embeddings[0])
            else:
                mean_embedding = np.mean(np.array(part_embeddings), axis=0)
                embeddings.append(mean_embedding.tolist())

            if idx % 50 == 0:
                print(f"Processed {idx}/{len(texts)} chunks")

        embeddings = np.array(embeddings, dtype=np.float32)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

embedding_manager = EmbeddingManager()
embedding_manager

In [18]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "all_text_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "Text document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: all_text_documents
Existing documents in collection: 0


In [21]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Processed 50/196 chunks
Processed 100/196 chunks
Processed 150/196 chunks
Generated embeddings with shape: (196, 384)
Adding 196 documents to vector store...
Successfully added 196 documents to vector store
Total documents in collection: 196


In [22]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [23]:
rag_retriever

In [24]:
rag_retriever.retrieve("In How many ways can I open a bank account?")

Retrieving documents for query: 'In How many ways can I open a bank account?'
Top K: 5, Score threshold: 0.0
Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_290362dc_177',
  'content': '**Q: What documents do I need to open a bank account?**\nA: Documentation requirements vary by country and bank type. For Indian accounts, you typically need Aadhaar card, PAN card, and address proof. For USA/UK accounts, passport and local address verification may be required. Our platform guides you through the specific requirements for each bank.\n\n**Q: How many accounts can I create?**\nA: There is no limit. Create as many as you need across different banks and countries.',
  'metadata': {'content_length': 477,
   'doc_index': 177,
   'source_file': 'USER_GUIDE.txt',
   'file_type': 'txt'},
  'similarity_score': 0.33140552043914795,
  'distance': 0.668594479560852,
  'rank': 1},
 {'id': 'doc_bffe6367_51',
  'content': '---\n\n## Account Creation\n\n**Q: How do I create a bank account?**\nA:\n1. Log in to your user account\n2. Go to "Browse Banks" to select a bank and country\n3. Click "Create Account" for your chosen bank\n4. Complete a 6-

In [25]:
rag_retriever.retrieve("In How many countries can I open a bank account?")

Retrieving documents for query: 'In How many countries can I open a bank account?'
Top K: 5, Score threshold: 0.0
Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_290362dc_177',
  'content': '**Q: What documents do I need to open a bank account?**\nA: Documentation requirements vary by country and bank type. For Indian accounts, you typically need Aadhaar card, PAN card, and address proof. For USA/UK accounts, passport and local address verification may be required. Our platform guides you through the specific requirements for each bank.\n\n**Q: How many accounts can I create?**\nA: There is no limit. Create as many as you need across different banks and countries.',
  'metadata': {'source_file': 'USER_GUIDE.txt',
   'content_length': 477,
   'file_type': 'txt',
   'doc_index': 177},
  'similarity_score': 0.30062222480773926,
  'distance': 0.6993777751922607,
  'rank': 1},
 {'id': 'doc_20523e04_54',
  'content': '---\n\n**Q: Can I create multiple accounts?**\nA:\nYes! You can create multiple accounts across different banks and countries. Each account is linked to your profile for easy management.\n\n*Also looking for: Multiple accou

RAG Pipeline- VectorDB To LLM Output Generation

In [26]:
import json

import requests

def ollama_rag_response(
    query, 
    rag_retriever, 
    ollama_url="http://localhost:11434/api/generate", 
    model_name="llama3.2:3b", 
    top_k=5
):
    # Step 1: Retrieve relevant context from vector DB
    retrieved_docs = rag_retriever.retrieve(query, top_k=top_k)
    context = "\n\n".join([doc['content'] for doc in retrieved_docs]) if retrieved_docs else ""
    if not context:
        return "No relevant context found to answer the question."

    # Step 2: Prepare prompt for LLM
    print("calling llm")
    prompt = f"""Context:\n{context}\n\nQuestion: {query}\nAnswer:"""
    print("loading data")
    # Step 3: Call Ollama LLM API (streaming)
    payload = {
        "model": model_name,
        "prompt": prompt
    }
    print("payload prepared, making API call...")
    response = requests.post(ollama_url, json=payload, stream=True)
    response.raise_for_status()
    answer = ""
    for line in response.iter_lines():
        if line:
            try:
                data = json.loads(line.decode("utf-8"))
                answer += data.get("response", "")
            except Exception:
                continue
    return answer if answer else "No response from LLM."

In [27]:
question = "On which countries can I open a bank account?"
response = ollama_rag_response(question, rag_retriever, model_name="llama3.2:3b")

Retrieving documents for query: 'On which countries can I open a bank account?'
Top K: 5, Score threshold: 0.0
Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)
calling llm
loading data
payload prepared, making API call...


In [28]:
print("Question:", question)
print("LLM Response:", response)

Question: On which countries can I open a bank account?
LLM Response: The supported countries for opening a bank account are:

* India
* USA
* UK

Additionally, the available locations may vary depending on the bank and country chosen.


In [29]:
question = "What Documents Do I need to open a bank account?"
response = ollama_rag_response(question, rag_retriever, model_name="llama3.2:3b")

Retrieving documents for query: 'What Documents Do I need to open a bank account?'
Top K: 5, Score threshold: 0.0
Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)
calling llm
loading data
payload prepared, making API call...


In [44]:
print("Question:", question)
print("LLM Response:", response)

Question: What Documents Do I need to open a bank account?
LLM Response: To open a bank account, you need the following documents:

1. Government-issued identity proof (e.g., Aadhaar for India, SSN for USA, NIN for UK)
2. Proof of address (e.g., utility bill, rental agreement, bank statement)
3. Proof of income (e.g., salary slip, tax return, employment letter)
4. Recent passport-size photograph
